# EDA & Problem Discovery

## Objective

This notebook explores the factors that influence demand and contribute to inventory-demand mismatches.

The analysis focuses on:

- Category
- Region
- Promotion
- Weather Condition
- Seasonality
- Epidemic
- Competitor Pricing
- Units Ordered
- Demand
- Inventory Level
- Potential lost sales

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/sales_data.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (76000, 16)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [7]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount',
       'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality',
       'Epidemic', 'Demand'],
      dtype='object')

## 1. Key Business Indicators

Before investigating individual factors, we define indicators that represent the main business problems.

### Inventory Shortage

Demand > Inventory Level

This indicates that available inventory may not have been sufficient to satisfy demand.

### Potential Lost Sales

Demand > Units Sold

This indicates that actual sales were lower than estimated demand and may represent unrealized sales.

### Demand-Sales Gap

Demand - Units Sold

Measures the amount of demand that was not converted into sales.

In [8]:
df["Inventory_Shortage"] = (
    df["Demand"] > df["Inventory Level"]
)

df["Potential_Lost_Sales"] = (
    df["Demand"] > df["Units Sold"]
)

print("Inventory shortage cases:",
      df["Inventory_Shortage"].sum())

print("Potential lost-sales cases:",
      df["Potential_Lost_Sales"].sum())

Inventory shortage cases: 10394
Potential lost-sales cases: 53440


## 2. Category Analysis

We investigate whether demand differs across product categories and whether some categories experience more inventory shortages than others.

In [9]:
category_demand = (
    df.groupby("Category")["Demand"]
      .agg(
          Average_Demand="mean",
          Median_Demand="median",
          Minimum_Demand="min",
          Maximum_Demand="max",
          Observations="count"
      )
      .sort_values("Average_Demand", ascending=False)
)

category_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand,Observations
Category,,,,,
Groceries,120.976447,116.0,4,430,30400
Clothing,112.619737,109.0,4,330,12160
Electronics,97.482018,95.0,4,339,9120
Toys,92.606955,92.0,4,361,10640
Furniture,73.581140,72.0,4,214,13680


In [10]:
category_shortage = (
    df.groupby("Category")["Inventory_Shortage"]
      .agg(
          Shortage_Cases="sum",
          Total_Observations="count"
      )
)

category_shortage["Shortage_Percentage"] = (
    category_shortage["Shortage_Cases"]
    / category_shortage["Total_Observations"]
    * 100
)

category_shortage.sort_values(
    "Shortage_Percentage",
    ascending=False
)

,Shortage_Cases,Total_Observations,Shortage_Percentage
Category,,,
Clothing,2362,12160,19.424342
Toys,1593,10640,14.971805
Electronics,1265,9120,13.870614
Groceries,4101,30400,13.490132
Furniture,1073,13680,7.843567


In [11]:
category_lost_sales = (
    df.groupby("Category")["Potential_Lost_Sales"]
      .agg(
          Lost_Sales_Cases="sum",
          Total_Observations="count"
      )
)

category_lost_sales["Lost_Sales_Percentage"] = (
    category_lost_sales["Lost_Sales_Cases"]
    / category_lost_sales["Total_Observations"]
    * 100
)

category_lost_sales.sort_values(
    "Lost_Sales_Percentage",
    ascending=False
)

,Lost_Sales_Cases,Total_Observations,Lost_Sales_Percentage
Category,,,
Clothing,8826,12160,72.582237
Groceries,21497,30400,70.713816
Toys,7507,10640,70.554511
Electronics,6430,9120,70.504386
Furniture,9180,13680,67.105263


## Category Analysis Findings

- Groceries have the highest average demand at approximately 120.98 units.
- Furniture has the lowest average demand at approximately 73.58 units.
- Clothing has the highest inventory shortage rate at approximately 19.42%.
- Clothing also has the highest potential lost-sales rate at approximately 72.58%.
- High demand does not necessarily correspond to the highest inventory shortage rate. For example, Groceries have the highest average demand, while Clothing has the highest shortage rate.
- This suggests that inventory mismatches may depend on factors beyond overall demand, such as promotions, seasonality, region, ordering behavior, weather, and product characteristics.

### Business Hypothesis

Some categories may require different inventory planning strategies because demand levels and inventory shortage risk do not necessarily move together.

## 3. Region Analysis

We investigate whether demand and inventory shortages vary across regions.

The objective is to determine whether certain regions have higher demand, greater inventory shortage risk, or higher potential lost-sales rates.

In [12]:
region_demand = (
    df.groupby("Region")["Demand"]
      .agg(
          Average_Demand="mean",
          Median_Demand="median",
          Minimum_Demand="min",
          Maximum_Demand="max",
          Observations="count"
      )
      .sort_values("Average_Demand", ascending=False)
)

region_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand,Observations
Region,,,,,
South,106.938882,103.0,4,351,15200
East,106.468684,102.0,4,361,15200
North,103.771645,100.0,4,430,30400
West,100.634934,96.0,4,370,15200


In [13]:
region_shortage = (
    df.groupby("Region")["Inventory_Shortage"]
      .agg(
          Shortage_Cases="sum",
          Total_Observations="count"
      )
)

region_shortage["Shortage_Percentage"] = (
    region_shortage["Shortage_Cases"]
    / region_shortage["Total_Observations"]
    * 100
)

region_shortage.sort_values(
    "Shortage_Percentage",
    ascending=False
)

,Shortage_Cases,Total_Observations,Shortage_Percentage
Region,,,
North,4438,30400,14.598684
East,2016,15200,13.263158
West,1975,15200,12.993421
South,1965,15200,12.927632


In [14]:
region_lost_sales = (
    df.groupby("Region")["Potential_Lost_Sales"]
      .agg(
          Lost_Sales_Cases="sum",
          Total_Observations="count"
      )
)

region_lost_sales["Lost_Sales_Percentage"] = (
    region_lost_sales["Lost_Sales_Cases"]
    / region_lost_sales["Total_Observations"]
    * 100
)

region_lost_sales.sort_values(
    "Lost_Sales_Percentage",
    ascending=False
)

,Lost_Sales_Cases,Total_Observations,Lost_Sales_Percentage
Region,,,
East,10804,15200,71.078947
North,21439,30400,70.523026
South,10606,15200,69.776316
West,10591,15200,69.677632


## 4. Promotion and Demand Analysis

We investigate whether promotions and discounts are associated with changes in demand.

In [15]:
promotion_demand = (
    df.groupby("Promotion")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Minimum_Demand="min",
        Maximum_Demand="max",
        Observations="count"
    )
)

promotion_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand,Observations
Promotion,,,,,
0,95.026843,92.0,4,330,51000
1,123.269400,120.0,4,430,25000


In [16]:
promotion_shortage = (
    df.groupby("Promotion")["Inventory_Shortage"]
    .agg(
        Shortage_Cases="sum",
        Total_Observations="count"
    )
)

promotion_shortage["Shortage_Percentage"] = (
    promotion_shortage["Shortage_Cases"]
    / promotion_shortage["Total_Observations"]
    * 100
)

promotion_shortage.sort_values(
    "Shortage_Percentage",
    ascending=False
)

,Shortage_Cases,Total_Observations,Shortage_Percentage
Promotion,,,
1,4889,25000,19.556000
0,5505,51000,10.794118


In [17]:
promotion_lost_sales = (
    df.groupby("Promotion")["Potential_Lost_Sales"]
    .agg(
        Lost_Sales_Cases="sum",
        Total_Observations="count"
    )
)

promotion_lost_sales["Lost_Sales_Percentage"] = (
    promotion_lost_sales["Lost_Sales_Cases"]
    / promotion_lost_sales["Total_Observations"]
    * 100
)

promotion_lost_sales.sort_values(
    "Lost_Sales_Percentage",
    ascending=False
)

,Lost_Sales_Cases,Total_Observations,Lost_Sales_Percentage
Promotion,,,
1,17983,25000,71.932000
0,35457,51000,69.523529


## 5. Weather and Demand Analysis

We investigate whether weather conditions are associated with changes in customer demand.

In [18]:
weather_demand = (
    df.groupby("Weather Condition")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Minimum_Demand="min",
        Maximum_Demand="max",
        Observations="count"
    )
    .sort_values("Average_Demand", ascending=False)
)

weather_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand,Observations
Weather Condition,,,,,
Sunny,115.167189,111.0,4,430,22980
Cloudy,105.404064,102.0,4,373,24360
Rainy,95.106743,92.0,4,378,17500
Snowy,94.045789,91.0,4,260,11160


## 6. Seasonality and Demand Analysis

We investigate whether seasonal conditions are associated with differences in demand.

In [19]:
seasonality_demand = (
    df.groupby("Seasonality")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Minimum_Demand="min",
        Maximum_Demand="max",
        Observations="count"
    )
    .sort_values("Average_Demand", ascending=False)
)

seasonality_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand,Observations
Seasonality,,,,,
Summer,112.855380,107.0,4,430,18400
Autumn,103.422967,100.0,4,327,18200
Winter,103.406190,100.0,4,339,21000
Spring,97.703098,95.0,4,361,18400


## 7. Epidemic Impact Analysis

We investigate whether epidemic conditions are associated with changes in demand.

In [20]:
epidemic_demand = (
    df.groupby("Epidemic")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Minimum_Demand="min",
        Maximum_Demand="max",
        Observations="count"
    )
)

epidemic_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand,Observations
Epidemic,,,,,
0,112.856743,108.0,4,430,60800
1,70.158816,64.0,4,279,15200


# 8. Key Findings from EDA

Based on the exploratory analysis, the following patterns were identified:

1. Demand varies significantly across product categories.
2. Clothing has the highest inventory shortage percentage among categories.
3. The North region has the highest inventory shortage percentage among regions.
4. Promotional periods have substantially higher average demand than non-promotional periods.
5. Sunny weather is associated with higher average demand than rainy or snowy conditions.
6. Summer has the highest average demand among the seasons.
7. Demand decreases significantly during epidemic periods.
8. Discount levels show a positive relationship with demand, although the correlation is moderate.
9. A substantial number of observations indicate potential inventory shortages and potential lost sales.
10. These factors should be considered when developing the demand forecasting model.

In [21]:
summary = pd.DataFrame({
    "Analysis": [
        "Category",
        "Region",
        "Promotion",
        "Weather",
        "Seasonality",
        "Epidemic",
        "Discount"
    ],
    "Key_Observation": [
        "Demand differs across product categories",
        "North has the highest shortage percentage",
        "Promotion increases average demand",
        "Sunny weather has the highest average demand",
        "Summer has the highest average demand",
        "Demand is substantially lower during epidemics",
        "Discount has a positive relationship with demand"
    ]
})

summary

,Analysis,Key_Observation
0,Category,Demand differs across product categories
1,Region,North has the highest shortage percentage
2,Promotion,Promotion increases average demand
3,Weather,Sunny weather has the highest average demand
4,Seasonality,Summer has the highest average demand
5,Epidemic,Demand is substantially lower during epidemics
6,Discount,Discount has a positive relationship with demand


## Conclusion

The EDA shows that demand is influenced by several business and external factors, including category, region, promotion, weather, seasonality, epidemic conditions, and discount.

The analysis also reveals significant inventory shortage and potential lost-sales patterns. Therefore, demand forecasting should not rely only on historical demand values. The forecasting model should incorporate relevant explanatory variables and temporal patterns.

These findings will guide the feature engineering and forecasting work in the next notebook.